# Model Training

### 1 - Library Import

In [ ]:
import sys
import os
sys.path.append(os.path.abspath(os.path.join('..')))
    
from src.data_processing import load_train_data
from src.data_processing import load_test_data
from src.model_evaluation import evaluate
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
import pandas as pd

### 2 - Data Prepping

In [ ]:
df_train = load_train_data("../CMAPSSData/train_FD001.txt")

df_test = load_test_data("../CMAPSSData/test_FD001.txt", "../CMAPSSData/RUL_FD001.txt")

In [ ]:
variance_table = df_train[df_train.columns[5:26]]
variance_table = variance_table.var()
variance_table = variance_table.to_frame(name='Variance')
variance_table = variance_table.round(5)
high_variance_features = variance_table[variance_table['Variance'] != 0].index.tolist()

In [ ]:
X_train = df_train[high_variance_features]
y_train = df_train['RUL']

X_test = df_test[high_variance_features]
y_test = df_test['RUL']

### 3 - Ridge

In [ ]:
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('ridge', Ridge(random_state=42))
])

groups = df_train['unit']
gkf = GroupKFold(n_splits=5)

# param_grid={'ridge__alpha': [i for i in range(0, 10000, 100)]}
# param_grid={'ridge__alpha': [i for i in range(400, 600, 10)]}
# param_grid={'ridge__alpha': [i for i in range(530, 550)]}
param_grid={'ridge__alpha': [545]}

ridge_cv = GridSearchCV(
    pipeline,
    param_grid=param_grid,  
    cv=gkf,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1   
)

ridge_cv.fit(X_train, y_train, groups=groups)

print(f"\nBest alpha : {ridge_cv.best_params_['ridge__alpha']}")
print(f"Best CV RMSE : {-ridge_cv.best_score_:.4f}")

In [ ]:
# All test rows
y_pred_all = ridge_cv.best_estimator_.predict(X_test)
evaluate(y_test, y_pred_all, label="All test rows")

# Official metric — last cycle per engine only
last_idx = df_test.groupby("unit")["cycle"].idxmax()
X_test_last = df_test.loc[last_idx, high_variance_features]
y_test_last = df_test.loc[last_idx, "RUL"]

y_pred_last = ridge_cv.best_estimator_.predict(X_test_last)
evaluate(y_test_last, y_pred_last, label="Last cycle per engine (official)")

In [ ]:
feature_names = high_variance_features
coefs = ridge_cv.best_estimator_.named_steps['ridge'].coef_

coef_df = pd.DataFrame({'feature': feature_names, 'coefficient': coefs})
coef_df = coef_df.reindex(coef_df['coefficient'].abs().sort_values(ascending=False).index)
print(coef_df)